## Import Datas

In [ ]:
import os
from google.colab import drive
drive.mount('/content/drive')
folder = "/content/drive/MyDrive/ICATH 2026/code/data/preprocessed options datas/"

Mounted at /content/drive


In [ ]:
import pandas as pd
pd.set_option('display.float_format', '{:.4f}'.format)
pd.set_option('display.max_columns', None)
import numpy as np
import random
import torch

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

In [ ]:
TechnologyUnderlying = [ "AAPL", "AMZN", "GOOGL", "META", "MSFT" ]
AutomobileUnderlying = [ "F", "GM", "LCID", "RIVN", "TSLA" ]

listTickers = AutomobileUnderlying + TechnologyUnderlying

In [ ]:
dataset = {
    "AAPL" : {"train" :pd.DataFrame(), "test" : pd.DataFrame()},
    "AMZN" : {"train" :pd.DataFrame(), "test" : pd.DataFrame()},
    "GOOGL" : {"train" :pd.DataFrame(), "test" : pd.DataFrame()},
    "META" : {"train" :pd.DataFrame(), "test" : pd.DataFrame()},
    "MSFT" : {"train" :pd.DataFrame(), "test" : pd.DataFrame()},
    "F" : {"train" :pd.DataFrame(), "test" : pd.DataFrame()},
    "GM" : {"train" :pd.DataFrame(), "test" : pd.DataFrame()},
    "LCID" : {"train" :pd.DataFrame(), "test" : pd.DataFrame()},
    "RIVN" : {"train" :pd.DataFrame(), "test" : pd.DataFrame()},
    "TSLA" : {"train" :pd.DataFrame(), "test" : pd.DataFrame()},
}

In [ ]:
for ticker in dataset.keys():
    train_path = os.path.join(folder, f"{ticker}_train.csv")
    test_path = os.path.join(folder, f"{ticker}_test.csv")

    if not os.path.exists(train_path):
        print(f"Missing: {train_path}")
        continue

    train_dataset = pd.read_csv(train_path)
    test_dataset = pd.read_csv(test_path)

    train_dataset = train_dataset.fillna(train_dataset.mean(numeric_only=True))
    test_dataset = test_dataset.fillna(test_dataset.mean(numeric_only=True))

    dataset[ticker]["train"] = train_dataset
    dataset[ticker]["test"] = test_dataset

## Define functions

In [ ]:
!pip install optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 419.5/419.5 kB 21.0 MB/s eta 0:00:00


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import numpy as np
from sklearn.metrics import mean_squared_error, mean_absolute_error
from sklearn.preprocessing import StandardScaler
import optuna

In [ ]:
test_size=0.2
random_state=42
target = 'lastPrice'
list_histos_datas_inputs = ["strike", "impliedVolatility", "timeToMaturity",
                            "riskFreeRate", "underlyingLastPrice"]

feature_combinations = [
    [],

    ["GS_" + ticker], ["SC_" + ticker], ["VIX"], ["PCR"],

    ["GS_" + ticker , "SC_" + ticker], ["GS_" + ticker , "VIX"], ["GS_" + ticker , "PCR"], ["SC_" + ticker , "VIX"], ["SC_" + ticker , "PCR"], ["VIX" , "PCR"],

    ["GS_" + ticker , "SC_" + ticker , "VIX"], ["GS_" + ticker , "SC_" + ticker , "PCR"], ["GS_" + ticker , "VIX" , "PCR"], ["SC_" + ticker , "VIX" , "PCR"],

	["GS_" + ticker , "SC_" + ticker , "VIX" , "PCR"]
]

In [ ]:
def build_ffnn(best_params, n_features):
    activations = {
        "relu": nn.ReLU,
        "tanh": nn.Tanh,
        "sigmoid": nn.Sigmoid,
        "elu": nn.ELU,
        "selu": nn.SELU
    }

    activation = activations[best_params["activation"]]
    n_layers = best_params["n_layers"]

    layers = []
    in_dim = n_features

    for i in range(n_layers):
        out_dim = best_params[f"n_units_{i}"]
        layers.append(nn.Linear(in_dim, out_dim))
        layers.append(activation())


        in_dim = out_dim

    layers.append(nn.Linear(in_dim, 1))
    return nn.Sequential(*layers)

def train_ffnn_with_optuna(X_train, y_train, X_test, y_test, best_params, epochs=50):

    X_train_t = torch.tensor(X_train, dtype=torch.float32)
    X_test_t  = torch.tensor(X_test,  dtype=torch.float32)
    y_train_t = torch.tensor(y_train, dtype=torch.float32).view(-1, 1)
    y_test_t  = torch.tensor(y_test,  dtype=torch.float32).view(-1, 1)

    dataset = TensorDataset(X_train_t, y_train_t)
    loader = DataLoader(dataset, batch_size=best_params["batch_size"], shuffle=True)

    model = build_ffnn(best_params, X_train.shape[1])

    optimizer = getattr(optim, best_params["optimizer"])(
        model.parameters(),
        lr=best_params["lr"]
    )

    loss_fn = nn.MSELoss()

    # --------- Training ----------
    for epoch in range(epochs):
        for batch_X, batch_y in loader:
            optimizer.zero_grad()
            pred = model(batch_X)
            loss = loss_fn(pred, batch_y)
            loss.backward()
            optimizer.step()

    # --------- Prediction ----------
    with torch.no_grad():
        y_pred = model(X_test_t).numpy()

    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    mae = mean_absolute_error(y_test, y_pred)
    nrmse = rmse / np.mean(y_test)

    return mae, rmse, nrmse


def create_model_for_ffnn(trial, n_features):

    n_layers = trial.suggest_int("n_layers", 2, 10)

    activation_name = trial.suggest_categorical(
        "activation", ["relu", "tanh", "sigmoid", "elu", "selu"]
    )

    activations = {
        "relu": nn.ReLU,
        "tanh": nn.Tanh,
        "sigmoid": nn.Sigmoid,
        "elu": nn.ELU,
        "selu": nn.SELU
    }
    activation = activations[activation_name]

    layers = []
    in_dim = n_features

    for i in range(n_layers):
        out_dim = trial.suggest_int(f"n_units_{i}", 2, 128)

        layers.append(nn.Linear(in_dim, out_dim))
        layers.append(activation())

        in_dim = out_dim

    # output layer
    layers.append(nn.Linear(in_dim, 1))

    return nn.Sequential(*layers)


# ---- objective ----
def objective_for_ffnn(trial, datasetForOptimization, X_train, y_train):

    model = create_model_for_ffnn(trial, X_train.shape[1])

    optimizer_name = trial.suggest_categorical(
        "optimizer", ["Adam", "AdamW", "SGD"] #optimizer
    )
    lr = trial.suggest_float("lr", 1e-5, 1e-2, log=True) #learning_rate

    batch_size = trial.suggest_categorical("batch_size", [32, 64, 128]) #batch_size

    optimizer = getattr(optim, optimizer_name)(model.parameters(), lr=lr)
    loader = DataLoader(datasetForOptimization, batch_size=batch_size, shuffle=False)

    loss_fn = nn.MSELoss()

    # ---- training ----
    for epoch in range(40):
        for batch_X, batch_y in loader:
            optimizer.zero_grad()
            pred = model(batch_X)
            loss = loss_fn(pred, batch_y)
            loss.backward()
            optimizer.step()

    return loss.item()

def predict_current_price_using_ffnn(option_type, ticker, list_histos_datas_inputs=list_histos_datas_inputs, feature_combinations=feature_combinations):
  for proxy in feature_combinations:
    #Prepare training dataset
    train_dataset = dataset[ticker]["train"]
    train_dataset = train_dataset[train_dataset["optionType"] == option_type]
    X_train = train_dataset[list_histos_datas_inputs + proxy].values
    y_train = train_dataset[["lastPrice"]].values

    #Prepare test dataset
    test_dataset = dataset[ticker]["test"]
    test_dataset = test_dataset[test_dataset["optionType"] == option_type]
    X_test = test_dataset[list_histos_datas_inputs+ proxy].values
    y_test = test_dataset[["lastPrice"]].values

    #Normaliser les dataset
    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train)
    X_test = scaler.fit_transform(X_test)

    #Determine best architecture and hyper-parameters
    X_tensor = torch.tensor(X_train, dtype=torch.float32)
    y_tensor = torch.tensor(y_train, dtype=torch.float32).view(-1, 1)
    datasetForOptimization = TensorDataset(X_tensor, y_tensor)

    study = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=SEED))
    study.optimize(lambda trial: objective_for_ffnn(trial, datasetForOptimization, X_train, y_train), n_trials=10)
    best_params = study.best_params
    best_params = study.best_params

    mae, rmse, nrmse = train_ffnn_with_optuna(
        X_train, y_train,
        X_test, y_test,
        best_params
    )
    print(f"{proxy} for {ticker} => (MAE={mae:.3f}; RMSE={rmse:.3f}; ; NRMSE={nrmse:.3f})")


## Price options using feed forward neural network

In [ ]:
ticker = "LCID"

In [ ]:
predict_current_price_using_ffnn("put", ticker)

[] for LCID => (MAE=0.322; RMSE=0.465; ; NRMSE=0.182)
['GS_LCID'] for LCID => (MAE=0.335; RMSE=0.449; ; NRMSE=0.175)
['SC_LCID'] for LCID => (MAE=0.337; RMSE=0.473; ; NRMSE=0.185)
['VIX'] for LCID => (MAE=0.364; RMSE=0.505; ; NRMSE=0.197)
['PCR'] for LCID => (MAE=0.345; RMSE=0.487; ; NRMSE=0.190)
['GS_LCID', 'SC_LCID'] for LCID => (MAE=0.377; RMSE=0.507; ; NRMSE=0.198)
['GS_LCID', 'VIX'] for LCID => (MAE=0.339; RMSE=0.471; ; NRMSE=0.184)
['GS_LCID', 'PCR'] for LCID => (MAE=0.327; RMSE=0.449; ; NRMSE=0.176)
['SC_LCID', 'VIX'] for LCID => (MAE=0.322; RMSE=0.456; ; NRMSE=0.178)
['SC_LCID', 'PCR'] for LCID => (MAE=0.334; RMSE=0.462; ; NRMSE=0.181)
['VIX', 'PCR'] for LCID => (MAE=0.332; RMSE=0.477; ; NRMSE=0.187)
['GS_LCID', 'SC_LCID', 'VIX'] for LCID => (MAE=0.350; RMSE=0.484; ; NRMSE=0.189)
['GS_LCID', 'SC_LCID', 'PCR'] for LCID => (MAE=0.318; RMSE=0.444; ; NRMSE=0.174)
['GS_LCID', 'VIX', 'PCR'] for LCID => (MAE=0.313; RMSE=0.448; ; NRMSE=0.175)
['SC_LCID', 'VIX', 'PCR'] for LCID => (MAE=

In [ ]:
predict_current_price_using_ffnn("call", ticker)

[] for LCID => (MAE=0.342; RMSE=0.499; ; NRMSE=0.209)
['GS_LCID'] for LCID => (MAE=0.294; RMSE=0.413; ; NRMSE=0.173)
['SC_LCID'] for LCID => (MAE=0.322; RMSE=0.443; ; NRMSE=0.185)
['VIX'] for LCID => (MAE=0.326; RMSE=0.489; ; NRMSE=0.205)
['PCR'] for LCID => (MAE=0.322; RMSE=0.478; ; NRMSE=0.200)
['GS_LCID', 'SC_LCID'] for LCID => (MAE=0.289; RMSE=0.395; ; NRMSE=0.165)
['GS_LCID', 'VIX'] for LCID => (MAE=0.316; RMSE=0.438; ; NRMSE=0.183)
['GS_LCID', 'PCR'] for LCID => (MAE=0.295; RMSE=0.407; ; NRMSE=0.170)
['SC_LCID', 'VIX'] for LCID => (MAE=0.290; RMSE=0.413; ; NRMSE=0.173)
['SC_LCID', 'PCR'] for LCID => (MAE=0.294; RMSE=0.414; ; NRMSE=0.173)
['VIX', 'PCR'] for LCID => (MAE=0.332; RMSE=0.485; ; NRMSE=0.203)
['GS_LCID', 'SC_LCID', 'VIX'] for LCID => (MAE=0.295; RMSE=0.408; ; NRMSE=0.171)
['GS_LCID', 'SC_LCID', 'PCR'] for LCID => (MAE=0.301; RMSE=0.410; ; NRMSE=0.172)
['GS_LCID', 'VIX', 'PCR'] for LCID => (MAE=0.299; RMSE=0.409; ; NRMSE=0.171)
['SC_LCID', 'VIX', 'PCR'] for LCID => (MAE=